# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will learn to:
- Load the Croissant dataset schema
- Overview available record sets, fields, and entity `@id`s
- Extract and analyze records
- Perform exploratory data analysis (EDA) and basic visualizations

### Dataset Source
The dataset is defined via a Croissant schema URL and describes the full metadata and data structure for:
> **Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution**

Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (run this cell once)
!pip install mlcroissant --quiet

## 1. Data Loading

Load the dataset metadata and records structure using `mlcroissant`. This allows programmatic access to the Croissant schema, record sets, and data fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the full metadata as a JSON-LD dict
metadata_dict = dataset.metadata.to_json()

print(f"Dataset Name: {dataset.metadata.name}")
print(f"Dataset Description: {dataset.metadata.description[:220]}...")

## 2. Data Overview

To explore the data, let's review the available **record sets**, their **fields**, and their unique identifiers (`@id`). In mlcroissant, record sets represent logical tables in the data package.

In [ ]:
# Discover record set @ids and basic info
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for recset in record_sets:
    print(f"- Record Set Name: {recset.name if hasattr(recset, 'name') else '[Unnamed]'}")
    print(f"  Record Set @id: {recset.id}")
    print(f"  Description: {recset.description if hasattr(recset, 'description') else ''}")
    # Print available field @ids
    print("  Fields:")
    for field in recset.fields:
        print(f"    - Field {field.name}: @id = {field.id}")
    print("")

## 3. Data Extraction

Let's extract all available record sets into individual Pandas DataFrames for inspection.

- Each record set is referenced **by its `@id`**.
- We'll show column names (field `@id`s) of each DataFrame.
- Demonstrate viewing the first few records from the main record set.

In [ ]:
# Collect all Record Set @ids
record_set_ids = [recset.id for recset in dataset.metadata.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        dataframes[recset_id] = pd.DataFrame(records)
    else:
        dataframes[recset_id] = pd.DataFrame()

# Display columns and preview of the first record set (if any rows present)
first_nonempty = None
for recset_id in record_set_ids:
    df = dataframes[recset_id]
    if not df.empty:
        first_nonempty = recset_id
        break

if first_nonempty:
    print(f"Record set {first_nonempty} columns:\n", dataframes[first_nonempty].columns.tolist())
    display(dataframes[first_nonempty].head())
else:
    print("No non-empty record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)

Let's perform basic EDA on a numeric field. The following cell demonstrates:
- Filtering for values above a threshold
- Normalizing the field
- Optionally grouping by a categorical field

Make sure to reference fields **by their `@id`** as specified by mlcroissant.

In [ ]:
# --- Please replace the following with true field @ids from section 2/3 as needed ---
# Select which record set and numeric field to analyze (use the printed @ids above):

record_set_id = first_nonempty   # Use the first non-empty record set by default

# Show available columns
print(f"Available columns in {record_set_id}:\n", dataframes[record_set_id].columns.tolist())

# For this dataset, suppose there is a numeric field like 'cr:Age' (replace with actual @id!)

# Auto-pick a numeric column (float or int) as example, fallback to user edit if none
import numpy as np
example_df = dataframes[record_set_id]
numeric_field = None
for col in example_df.columns:
    if np.issubdtype(example_df[col].dtype, np.number):
        numeric_field = col
        break
if numeric_field is None:
    # Try to pick a field name containing 'age' or 'interval'
    for col in example_df.columns:
        if ('age' in col.lower()) or ('interval' in col.lower()):
            numeric_field = col
            break

if numeric_field is None:
    print("No numeric field found automatically. Please specify a field @id (column name) from the list above as `numeric_field`.")
else:
    print(f"Using numeric field: {numeric_field}")
    threshold = example_df[numeric_field].mean() if example_df[numeric_field].notnull().any() else 0
    threshold = threshold if not np.isnan(threshold) else 0
    filtered_df = example_df[example_df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a categorical column to group by (e.g., one with dtype 'object', not the numeric field)
    group_field = None
    for col in example_df.columns:
        if col != numeric_field and example_df[col].dtype == object:
            group_field = col
            break
    if group_field:
        print(f"Grouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        display(grouped_df.head())
    else:
        print("No categorical field available for grouping.")

## 5. Visualization

Visualize the distribution of the chosen numeric field, and its relationship to the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution
if numeric_field and not example_df[numeric_field].isnull().all():
    plt.figure(figsize=(7,4))
    sns.histplot(example_df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

if (numeric_field is not None) and (group_field is not None):
    plt.figure(figsize=(8,5))
    sns.boxplot(y=example_df[numeric_field], x=example_df[group_field])
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

- We demonstrated how to load clinical dataset metadata via Croissant with `mlcroissant`
- Listed record sets, fields, `@id` references, and loaded DataFrames
- Applied simple filtering, normalization, and visual exploration of a numeric variable
- This workflow may be adapted to explore any Croissant/FAIR-harmonized dataset in biomedical or scientific research.

**For advanced analysis, repeat the above with other record sets, alter filter/group/plot logic, or integrate predictive modeling.**